## Loss Comparison
Compares CE, dice, and CE+dice losses across training and validation metrics.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Config
# Set path to results directory
K = 4
RUNS = {
    "CE":                   Path("../results/segthor/ce"),
    "Dice":                 Path("../results/segthor/dice"),
    "CE + Dice":            Path("../results/segthor/ce_dice"),
    "Weighted Dice":        Path("../results/segthor/w_dice"),
    "CE + Weighted Dice":   Path("../results/segthor/ce_w_dice"),
}

CLASS_NAMES = ["Background", "Esophagus+Aorta", "Heart", "Trachea"]
COLORS = plt.cm.tab10(np.linspace(0, 1, len(RUNS)))


In [ ]:
# Load results
def load(dest: Path) -> dict:
    return {
        "loss_tra": np.load(dest / "loss_tra.npy"),
        "loss_val": np.load(dest / "loss_val.npy"),
        "dice_tra": np.load(dest / "dice_tra.npy"),
        "dice_val": np.load(dest / "dice_val.npy"),
    }

data = {name: load(path) for name, path in RUNS.items()}
epochs = next(iter(data.values()))["loss_tra"].shape[0]
xs = range(epochs)

# Print a summary of every run
run_width = max(len("Run"), max(len(name) for name in data)) + 2
header = (
    f"{'Run':<{run_width}}"
    f"{'Epochs':>10}"
    f"{'Train samples':>15}"
    f"{'Val samples':>13}"
    f"{'Classes':>9}"
)

print(header)
print("-" * len(header))

for name, d in data.items():
    epochs_run = d["loss_tra"].shape[0]
    train_samp = d["dice_tra"].shape[1]
    val_samp   = d["dice_val"].shape[1]
    n_classes  = d["dice_tra"].shape[2]

    print(
        f"{name:<{run_width}}"
        f"{epochs_run:>10}"
        f"{train_samp:>15}"
        f"{val_samp:>13}"
        f"{n_classes:>9}"
    )

In [ ]:
# Loss curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Loss curves (note: CE and Dice are on different scales)")

for ax, split, key in zip(axes, ["Training", "Validation"], ["loss_tra", "loss_val"]):
    for (name, d), color in zip(data.items(), COLORS):
        ax.plot(xs, d[key].mean(axis=1), label=name, color=color)
    ax.set_title(f"{split} Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Mean foreground Dice with best epoch
fig, ax = plt.subplots(figsize=(10, 4))

for (name, d), color in zip(data.items(), COLORS):
    mean_dice = d["dice_val"][:, :, 1:].mean(axis=(1, 2))
    ax.plot(xs, mean_dice, label=name, color=color)

    best_epoch = int(np.argmax(mean_dice))
    best_dice  = mean_dice[best_epoch]
    ax.axvline(best_epoch, color=color, linestyle='--', alpha=0.5)
    ax.scatter(best_epoch, best_dice, color=color, zorder=5)
    ax.annotate(f"{best_dice:.3f}",
                xy=(best_epoch, best_dice),
                xytext=(best_epoch + 0.5, best_dice - 0.02),
                fontsize=8, color=color)

ax.set_title("Mean Foreground Dice with best epoch markers")
ax.set_xlabel("Epoch")
ax.set_ylabel("DSC")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()